# Person A — Notebook 1 v2.1: ACDC Circuit Discovery (threshold-fixed)
**CS 590NN | Amogh | Apr 18 rerun**

**Changes vs v2:**
1. `set_use_attn_result(True)` after model load (from v2 — attn heads are scorable).
2. **Threshold raised 0.1 → 0.3.** Standard ACDC threshold; v2 with 0.1 produced circuits averaging 90.7 heads per sample (pathological — max was 425 / 448 total heads).
3. **Effect-range denominator floored at 0.5.** The normalized causal score is `|Δlogit_diff| / effect_range`. When `effect_range` is near zero (i.e., model has no strong prior on target_true vs target_new to begin with), the denominator collapses and every head looks causal. Floor at 0.5 means low-signal samples don't inflate circuit size.
4. Per-sample `effect_clamped` flag records which samples hit the floor — so we can audit later.

**Output:** `week2_circuit_log_v2.1.json` (preserves both v1 and v2 logs for diff).

**Runtime:** ~20 min on A100. Same as v2.


### 1.0 Install
Run once. Runtime restarts automatically — do not re-run after restart.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens","transformers","datasets","accelerate","einops","jaxtyping"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)

### 1.1 Imports
Start here after restart.

In [ ]:
import torch, json, re, requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer, ActivationCache

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2,0),     "NumPy >= 2.0 — re-run cell 1.0 and restart."
print(f"NumPy  : {np.__version__}")

torch.manual_seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Imports OK")

### 1.2 Load Model

In [ ]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

# v2 fix: enable hook_result so ACDC can score attention heads
model.set_use_attn_result(True)

model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, total = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Config : {model.cfg.n_layers} layers | {model.cfg.n_heads} heads | d_model={model.cfg.d_model}")
print(f"Memory : {free/1e9:.1f} GB free after load")

### 1.3 Native ACDC Implementation

In [ ]:
class NativeACDC:
    """
    ACDC-style threshold-based circuit discovery via activation patching.
    Conmy et al., NeurIPS 2023. Implemented natively via TransformerLens.
    Activation caches are deleted after each sample to keep GPU memory flat.

    v2.1 changes:
      - Threshold default raised 0.1 -> 0.3 (tighter circuits, fewer false positives).
      - Effect-range denominator floored at 0.5 so samples where the model has
        no strong prior on target_true vs target_new don't get denominator
        collapse. Per-sample `effect_clamped` flag records which samples hit
        the floor.
    """
    def __init__(self, model, threshold=0.3, effect_floor=0.5):
        self.model        = model
        self.threshold    = threshold
        self.effect_floor = effect_floor

    def _logit_diff(self, logits, target_id, baseline_id):
        return (logits[0, -1, target_id] - logits[0, -1, baseline_id]).item()

    def run(self, prompt, target_true, target_new):
        true_id = self.model.to_tokens(target_true, prepend_bos=False)[0, 0].item()
        new_id  = self.model.to_tokens(target_new,  prepend_bos=False)[0, 0].item()
        tokens  = self.model.to_tokens(prompt)

        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, self.model.cfg.d_vocab-1, (tokens.shape[1]-2,))

        with torch.no_grad():
            clean_logits,   clean_cache   = self.model.run_with_cache(tokens)
            corrupt_logits, corrupt_cache = self.model.run_with_cache(corrupt)

        clean_score   = self._logit_diff(clean_logits,   true_id, new_id)
        corrupt_score = self._logit_diff(corrupt_logits, true_id, new_id)
        raw_effect    = abs(clean_score - corrupt_score)
        effect_range  = max(raw_effect, self.effect_floor)
        effect_clamped = raw_effect < self.effect_floor

        causal_scores = {}

        for layer in range(self.model.cfg.n_layers):
            # Attention heads
            hn = f"blocks.{layer}.attn.hook_result"
            if hn in clean_cache:
                for head in range(self.model.cfg.n_heads):
                    def make_attn(h=head, n=hn):
                        ca = corrupt_cache[n][:, :, h:h+1, :].clone()
                        def fn(v, hook): v[:, :, h:h+1, :] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_attn())])
                    causal_scores[f"attn_{layer}_{head}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

            # MLP layers
            hn = f"blocks.{layer}.hook_mlp_out"
            if hn in clean_cache:
                def make_mlp(n=hn):
                    ca = corrupt_cache[n].clone()
                    def fn(v, hook): return ca
                    return fn
                with torch.no_grad():
                    pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_mlp())])
                causal_scores[f"mlp_{layer}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

        # Free caches immediately
        del clean_cache, corrupt_cache, clean_logits, corrupt_logits
        torch.cuda.empty_cache()

        attn_heads, mlp_layers = [], []
        for node, score in causal_scores.items():
            if score >= self.threshold:
                parts = node.split("_")
                if parts[0] == "attn": attn_heads.append((int(parts[1]), int(parts[2])))
                else:                  mlp_layers.append(int(parts[1]))

        return {
            "attn_heads":     sorted(set(attn_heads)),
            "mlp_layers":     sorted(set(mlp_layers)),
            "causal_scores":  causal_scores,
            "clean_score":    clean_score,
            "corrupt_score":  corrupt_score,
            "raw_effect":     raw_effect,
            "effect_clamped": effect_clamped,
        }

print("NativeACDC (v2.1) defined. Threshold=0.3, effect_floor=0.5.")


### 1.4 Load CounterFact

In [ ]:
@dataclass
class EditSample:
    prompt:          str
    target_new:      str
    target_true:     str
    related_prompts: List[str] = field(default_factory=list)

print("Downloading CounterFact from ROME project source...")
raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
print(f"Downloaded {len(raw_data)} records")

N_SAMPLES = 50

def parse_counterfact(raw):
    neighborhood = raw.get("neighborhood_prompts", [])
    related = [p for p in neighborhood[:3] if isinstance(p, str)]
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" "  + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=related,
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples")
print(f"Example: '{cf_samples[0].prompt}' -> true='{cf_samples[0].target_true}'")

### 1.5 Run ACDC on 50 Samples

In [ ]:
acdc        = NativeACDC(model, threshold=0.3, effect_floor=0.5)
circuit_log = []

print(f"Running ACDC v2.1 on {N_SAMPLES} samples...")
print(f"  threshold    = {acdc.threshold}")
print(f"  effect_floor = {acdc.effect_floor}")
print("Caches freed after each sample.\n")

for i, s in enumerate(cf_samples):
    result = acdc.run(s.prompt, s.target_true, s.target_new)
    circuit_log.append({
        "idx":             i,
        "prompt":          s.prompt,
        "attn_heads":      result["attn_heads"],
        "mlp_layers":      result["mlp_layers"],
        "clean_score":     round(result["clean_score"],    4),
        "corrupt_score":   round(result["corrupt_score"],  4),
        "raw_effect":      round(result["raw_effect"],     4),
        "effect_clamped":  bool(result["effect_clamped"]),
        "n_attn":          len(result["attn_heads"]),
        "n_mlp":           len(result["mlp_layers"]),
    })
    if i % 10 == 0:
        free = torch.cuda.mem_get_info()[0] / 1e9
        clamp = " [CLAMPED]" if result["effect_clamped"] else ""
        print(f"  [{i+1:>2}/{N_SAMPLES}]  n_attn={len(result['attn_heads']):>3}  "
              f"n_mlp={len(result['mlp_layers']):>2}  "
              f"raw_eff={result['raw_effect']:.2f}{clamp}  gpu_free={free:.1f}GB")

with open(RESULTS_DIR / "week2_circuit_log_v2.1.json", "w") as f:
    json.dump(circuit_log, f, indent=2)

from collections import Counter
all_attn = [str(h) for e in circuit_log for h in e["attn_heads"]]
all_mlp  = [str(l) for e in circuit_log for l in e["mlp_layers"]]
print(f"\nTop attention heads : {Counter(all_attn).most_common(5)}")
print(f"Top MLP layers      : {Counter(all_mlp).most_common(5)}")
print(f"Saved -> {RESULTS_DIR}/week2_circuit_log_v2.1.json")


### 1.6 Save and Download Results

In [ ]:
import shutil
from datetime import datetime
from collections import Counter

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Core circuit size stats
n_attn_list = [e["n_attn"] for e in circuit_log]
n_mlp_list  = [e["n_mlp"]  for e in circuit_log]
n_clamped   = sum(1 for e in circuit_log if e["effect_clamped"])

# Circuit size distribution
buckets_def = [("<=20", 20), ("21-50", 50), ("51-100", 100),
               ("101-200", 200), ("201-300", 300), (">300", 10**9)]
dist = {}
prev = 0
for label, upper in buckets_def:
    dist[label] = sum(1 for n in n_attn_list if prev < n <= upper)
    prev = upper

# Sanity check: effect-range / n_attn correlation should be near zero after fix
# (it was -0.46 in v2 — threshold collapse was inflating circuits)
def pearson(xs, ys):
    mx, my = sum(xs)/len(xs), sum(ys)/len(ys)
    num = sum((x-mx)*(y-my) for x,y in zip(xs, ys))
    den = (sum((x-mx)**2 for x in xs) * sum((y-my)**2 for y in ys)) ** 0.5
    return num/den if den else 0.0

effect_vs_nattn = pearson([e["raw_effect"] for e in circuit_log], n_attn_list)

summary = {
    "author":           "Amogh",
    "notebook":         "Notebook 1 v2.1 — ACDC Circuit Discovery (threshold-fixed)",
    "model":            MODEL_NAME,
    "timestamp":        timestamp,
    "n_samples":        N_SAMPLES,
    "threshold":        acdc.threshold,
    "effect_floor":     acdc.effect_floor,
    "circuit_summary": {
        "avg_attn_per_sample": round(sum(n_attn_list) / N_SAMPLES, 2),
        "avg_mlp_per_sample":  round(sum(n_mlp_list)  / N_SAMPLES, 2),
        "median_n_attn":       sorted(n_attn_list)[N_SAMPLES//2],
        "median_n_mlp":        sorted(n_mlp_list)[N_SAMPLES//2],
        "max_n_attn":          max(n_attn_list),
        "max_n_mlp":           max(n_mlp_list),
        "top_attn_heads":      Counter(all_attn).most_common(5),
        "top_mlp_layers":      Counter(all_mlp).most_common(5),
        "circuit_size_distribution": dist,
        "n_samples_effect_clamped":  n_clamped,
        "effect_vs_nattn_pearson":   round(effect_vs_nattn, 3),
    }
}

with open(RESULTS_DIR / f"summary_nb1v2.1_{timestamp}.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = f"/content/PersonA_Notebook1v2.1_results_{timestamp}"
shutil.make_archive(zip_path, "zip", RESULTS_DIR)

print("=" * 65)
print(f"  NOTEBOOK 1 v2.1 RESULTS — Amogh")
print(f"  {timestamp}")
print("=" * 65)
print(f"  Config              : threshold={acdc.threshold}  effect_floor={acdc.effect_floor}")
print(f"  Avg attn/sample     : {summary['circuit_summary']['avg_attn_per_sample']}  (was 90.7 in v2)")
print(f"  Median attn         : {summary['circuit_summary']['median_n_attn']}")
print(f"  Max attn            : {summary['circuit_summary']['max_n_attn']}  (was 425 in v2)")
print(f"  Avg MLP/sample      : {summary['circuit_summary']['avg_mlp_per_sample']}")
print(f"  Samples effect-clamped : {n_clamped} / {N_SAMPLES}")
print(f"  Pearson(effect, n_attn): {effect_vs_nattn:+.3f}  (was -0.46 in v2; ~0 is ideal)")
print()
print("  Circuit size distribution:")
for k, v in dist.items():
    bar = "#" * v
    print(f"    {k:<10} : {v:>2}  {bar}")
print()
print("  Top heads           :", summary["circuit_summary"]["top_attn_heads"])
print()
print("  Files saved:")
for fp in sorted(RESULTS_DIR.glob("*.json")):
    print(f"    {fp.name:<40}  {fp.stat().st_size//1024:>4} KB")
print(f"\n  Download: {zip_path}.zip")
print("=" * 65)

# ── Sanity assertions — will print warnings, not raise, so you can inspect ──
if summary["circuit_summary"]["avg_attn_per_sample"] > 50:
    print("\n  WARNING: avg n_attn still > 50. Consider threshold=0.4 or effect_floor=1.0.")
elif summary["circuit_summary"]["avg_attn_per_sample"] < 3:
    print("\n  WARNING: avg n_attn very low (< 3). Threshold may be too strict.")
else:
    print("\n  Circuit sizes look reasonable. Upload week2_circuit_log_v2.1.json to Notebook 3.")

from google.colab import files
files.download(f"{zip_path}.zip")
